In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# =========================
# 1. Imports
# =========================
import os
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, classification_report

# =========================
# 2. LOAD DATASET
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/codelama-devignx26"

df = None
for root, dirs, files in os.walk(base_path):
    for f in files:
        if f.endswith(".csv"):
            df = pd.read_csv(os.path.join(root, f))
            print("✅ Loaded:", f)
            break
    if df is not None:
        break

if df is None:
    raise Exception("Dataset not found")

df = df[['code', 'label']].dropna()
df.columns = ['text', 'label']

# ⚠️ LLM is slow → use subset
df = df.sample(200, random_state=42)

texts = df['text'].astype(str).tolist()
labels = df['label'].astype(int).tolist()

print("Samples:", len(texts))

# =========================
# 3. LOAD MODEL
# =========================
model_name = "codellama/CodeLlama-7b-Instruct-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

# =========================
# 4. PROMPT
# =========================
def build_prompt(code):
    return f"""
You are a cybersecurity expert.

Your task: classify the following code.

If the code contains any vulnerability → output 1  
If the code is safe → output 0  

IMPORTANT:
- Output ONLY a single number: 0 or 1
- Do NOT explain
- Do NOT write anything else

Code:
{code}

Answer:
"""

# =========================
# 5. PREDICTION FUNCTION (FIXED)
# =========================
def predict(code):
    prompt = build_prompt(code)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        temperature=0.0
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 🔥 CLEAN PARSING
    answer = response.split("Answer:")[-1].strip()

    if answer.startswith("1"):
        return 1
    elif answer.startswith("0"):
        return 0
    else:
        return 0   # fallback safety

# =========================
# 6. RUN INFERENCE
# =========================
preds = []

print("\n🔍 Running CodeLlama inference...")

for code in tqdm(texts):
    try:
        pred = predict(code[:800])   # 🔥 limit length
    except:
        pred = 0
    preds.append(pred)

# =========================
# 7. EVALUATION
# =========================
acc = accuracy_score(labels, preds)

print("\n✅ Accuracy:", acc)
print("\n📊 Classification Report:\n", classification_report(labels, preds))

# =========================
# 8. SAVE RESULTS
# =========================
pd.DataFrame({
    "true_label": labels,
    "predicted_label": preds
}).to_csv("/kaggle/working/codellama_predictions.csv", index=False)

print("\n✅ Predictions saved")

✅ Loaded: Devignx_validation.csv
Samples: 200


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


🔍 Running CodeLlama inference...



  0%|          | 0/200 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  0%|          | 1/200 [00:00<01:07,  2.96it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  1%|          | 2/200 [00:01<01:58,  1.67it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  2%|▏         | 3/200 [00:01<01:53,  1.74it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  2%|▏         | 4/200 [00:02<01:43,  1.89it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  2%|▎         | 5/200 [00:02<01:53,  1.71it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  3%|▎         | 6/200 [00:03<02:04,  1.56it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  4%|▎         | 7/200 [00:04<01:59,  1.61it/s]Setting `pad_t


✅ Accuracy: 0.535

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.60      0.63      0.61       118
           1       0.43      0.40      0.42        82

    accuracy                           0.54       200
   macro avg       0.52      0.51      0.51       200
weighted avg       0.53      0.54      0.53       200


✅ Predictions saved
